In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

In [2]:


# --- 1. Challenge Parameters ---
S0 = 50         # UPDATE THIS: Current price of AETHER_CRYSTAL
sigma = 2.51
TRADING_DAYS_PER_YEAR = 252
STEPS_PER_DAY = 4
STEPS_PER_YEAR = TRADING_DAYS_PER_YEAR * STEPS_PER_DAY
dt = 1.0 / STEPS_PER_YEAR
NUM_SIMULATIONS = 100

# Calculate discrete steps for expiries
STEPS_2W = int(round(2 * 5 * STEPS_PER_DAY)) 
STEPS_3W = int(round(3 * 5 * STEPS_PER_DAY)) 

# --- 2. Simulate 100 GBM Paths ---
np.random.seed(67)

Z = np.random.standard_normal((NUM_SIMULATIONS, STEPS_3W))
drift = (- 0.5 * sigma**2) * dt
diffusion = sigma * np.sqrt(dt) * Z
log_returns = drift + diffusion

# Initialize price matrix: Shape (100 paths, 61 steps including t=0)
prices = np.zeros((NUM_SIMULATIONS, STEPS_3W + 1))
prices[:, 0] = S0

for t in range(1, STEPS_3W + 1):
    prices[:, t] = prices[:, t-1] * np.exp(log_returns[:, t-1])

# Isolate prices at specific expiry points
prices_2w = prices[:, STEPS_2W]
prices_3w = prices[:, STEPS_3W]


# --- 4. Payoff Calculations ---
payoffs = {}

# Vanilla Call & Put (3 Weeks)
payoffs["AC_50_P"] = np.maximum(50 - prices_3w, 0)
payoffs["AC_50_C"] = np.maximum(prices_3w - 50, 0)
payoffs["AC_35_P"] = np.maximum(35 - prices_3w, 0)
payoffs["AC_40_P"] = np.maximum(40 - prices_3w, 0)
payoffs["AC_45_P"] = np.maximum(45 - prices_3w, 0)
payoffs["AC_60_C"] = np.maximum(prices_3w - 60, 0)

# Vanilla Call & Put (2 Weeks)
payoffs["AC_50_P_2"] = np.maximum(50 - prices_2w, 0)
payoffs["AC_50_C_2"] = np.maximum(prices_2w - 50, 0)

# Chooser Option 
# At 2 weeks (Step 40), it becomes a Call if S > K, or a Put if S <= K.
is_call = prices_2w > 50
payoffs["AC_50_CO"] = np.where(is_call, 
                          np.maximum(prices_3w - 50, 0), 
                          np.maximum(50 - prices_3w, 0))

# Binary Put Option (3 Weeks)
payoffs["AC_40_BP"] = np.where(prices_3w < 40, 10, 0)

# Knock-Out Put Option (3 Weeks)
# Checks if the price dropped below the barrier at ANY discrete step
min_prices_3w = np.min(prices[:, :STEPS_3W+1], axis=1)
payoffs["AC_35_KO"] = np.where(min_prices_3w < 35, 
                                0.0, 
                                np.maximum(45 - prices_3w, 0))

# --- 5. Final Fair Value Outputs ---
print("--- FAIR VALUES (Average over 100 simulations) ---")
for option, payoff in payoffs.items():
    print(f"{option}: {np.mean(payoff):.4f}")

--- FAIR VALUES (Average over 100 simulations) ---
AC_50_P: 11.9039
AC_50_C: 12.1264
AC_35_P: 4.3427
AC_40_P: 6.3206
AC_45_P: 8.8565
AC_60_C: 8.8994
AC_50_P_2: 9.5240
AC_50_C_2: 8.5218
AC_50_CO: 21.9052
AC_40_BP: 4.3000
AC_35_KO: 0.2249


In [3]:
import numpy as np

# --- 1. Challenge Parameters ---
S0 = 50
sigma = 2.51
TRADING_DAYS_PER_YEAR = 252
STEPS_PER_DAY = 4
STEPS_PER_YEAR = TRADING_DAYS_PER_YEAR * STEPS_PER_DAY
dt = 1.0 / STEPS_PER_YEAR
NUM_SIMULATIONS = 100

STEPS_2W = int(round(2 * 5 * STEPS_PER_DAY)) 
STEPS_3W = int(round(3 * 5 * STEPS_PER_DAY)) 

# --- 2. Simulate 100 GBM Paths ---
np.random.seed(67)

Z = np.random.standard_normal((NUM_SIMULATIONS, STEPS_3W))
drift = (- 0.5 * sigma**2) * dt
diffusion = sigma * np.sqrt(dt) * Z
log_returns = drift + diffusion

prices = np.zeros((NUM_SIMULATIONS, STEPS_3W + 1))
prices[:, 0] = S0

for t in range(1, STEPS_3W + 1):
    prices[:, t] = prices[:, t-1] * np.exp(log_returns[:, t-1])

prices_2w = prices[:, STEPS_2W]
prices_3w = prices[:, STEPS_3W]

# --- 3. Payoff Calculations ---
payoffs = {}
payoffs["AC_50_P"] = np.maximum(50 - prices_3w, 0)
payoffs["AC_50_C"] = np.maximum(prices_3w - 50, 0)
payoffs["AC_35_P"] = np.maximum(35 - prices_3w, 0)
payoffs["AC_40_P"] = np.maximum(40 - prices_3w, 0)
payoffs["AC_45_P"] = np.maximum(45 - prices_3w, 0)
payoffs["AC_60_C"] = np.maximum(prices_3w - 60, 0)
payoffs["AC_50_P_2"] = np.maximum(50 - prices_2w, 0)
payoffs["AC_50_C_2"] = np.maximum(prices_2w - 50, 0)

# --- 4. Market Prices (Bid, Ask) ---
market = {
    "AC_50_P": (12.00, 12.05),
    "AC_50_C": (12.00, 12.05),
    "AC_35_P": (4.33, 4.35),
    "AC_40_P": (6.50, 6.55),
    "AC_45_P": (9.05, 9.10),
    "AC_60_C": (8.80, 8.85),
    "AC_50_P_2": (9.70, 9.75),
    "AC_50_C_2": (9.70, 9.75)
}

# --- 5. Strategies & Gap Analysis ---
strategies = {
    "Long Straddle": {
        "legs": [("AC_50_C", 1), ("AC_50_P", 1)],
        "description": "Long Call + Long Put (Same Strike/Expiry)"
    },
    "Long Strangle": {
        "legs": [("AC_60_C", 1), ("AC_45_P", 1)],
        "description": "Long OTM Call + Long OTM Put"
    },
    "Bull Call Spread": {
        "legs": [("AC_50_C", 1), ("AC_60_C", -1)],
        "description": "Long Call + Short Call (Higher Strike)"
    },
    "Bear Put Spread": {
        "legs": [("AC_50_P", 1), ("AC_45_P", -1)],
        "description": "Long Put + Short Put (Lower Strike)"
    },
    "Calendar Call Spread": {
        "legs": [("AC_50_C", 1), ("AC_50_C_2", -1)],
        "description": "Long Long-Term Call + Short Short-Term Call"
    },
    "Calendar Put Spread": {
        "legs": [("AC_50_P", 1), ("AC_50_P_2", -1)],
        "description": "Long Long-Term Put + Short Short-Term Put"
    },
    "Long Put Butterfly": {
        "legs": [("AC_50_P", 1), ("AC_45_P", -2), ("AC_40_P", 1)],
        "description": "Long ITM Put + 2x Short ATM Put + Long OTM Put"
    }
}

print(f"{'Strategy Name':<25} | {'Market Cost':<12} | {'Fair Value':<12} | {'Gap (Fair - Market)':<20}")
print("-" * 75)

for name, details in strategies.items():
    market_cost = 0.0
    fair_value = np.zeros(NUM_SIMULATIONS)
    
    for option, qty in details["legs"]:
        # Calculate Market Cost (Buy at Ask, Sell at Bid)
        if qty > 0:
            market_cost += qty * market[option][1] 
        else:
            market_cost += qty * market[option][0] 
            
        # Calculate Fair Value array
        fair_value += qty * payoffs[option]
    
    mean_fair_value = np.mean(fair_value)
    gap = mean_fair_value - market_cost
    
    print(f"{name:<25} | {market_cost:<12.3f} | {mean_fair_value:<12.3f} | {gap:<20.3f}")

Strategy Name             | Market Cost  | Fair Value   | Gap (Fair - Market) 
---------------------------------------------------------------------------
Long Straddle             | 24.100       | 24.030       | -0.070              
Long Strangle             | 17.950       | 17.756       | -0.194              
Bull Call Spread          | 3.250        | 3.227        | -0.023              
Bear Put Spread           | 3.000        | 3.047        | 0.047               
Calendar Call Spread      | 2.350        | 3.605        | 1.255               
Calendar Put Spread       | 2.350        | 2.380        | 0.030               
Long Put Butterfly        | 0.500        | 0.512        | 0.012               


In [5]:
#finding the expected profit we can make for each product
# if our fair value is higher than the market price, we can make a profit by buying the product and selling it at the market price
# if our fair value is lower than the market price, we can make a profit by selling the product and buying it at the market price
# display expected profit for each product

for product in market:
    market_price = (market[product][0] + market[product][1]) / 2
    fair_value = np.mean(payoffs[product])
    if fair_value > market_price:
        print(f"buying {product} at market price {market_price:.2f}, fv {fair_value:.2f}, spread = {abs(market_price - fair_value):.2f}")
    elif fair_value < market_price:
        print(f"selling {product} at market price {market_price:.2f}, fv {fair_value:.2f}, spread = {abs(market_price - fair_value):.2f}")


selling AC_50_P at market price 12.03, fv 11.90, spread = 0.12
buying AC_50_C at market price 12.03, fv 12.13, spread = 0.10
buying AC_35_P at market price 4.34, fv 4.34, spread = 0.00
selling AC_40_P at market price 6.53, fv 6.32, spread = 0.20
selling AC_45_P at market price 9.07, fv 8.86, spread = 0.22
buying AC_60_C at market price 8.82, fv 8.90, spread = 0.07
selling AC_50_P_2 at market price 9.72, fv 9.52, spread = 0.20
selling AC_50_C_2 at market price 9.72, fv 8.52, spread = 1.20


In [ ]:
import numpy as np

# --- 1. Challenge Parameters ---
def options_pricing(S0=50, sigma=2.51, num_simulations=1000):
    S0 = S0
    sigma = sigma
    TRADING_DAYS_PER_YEAR = 252
    STEPS_PER_DAY = 4
    STEPS_PER_YEAR = TRADING_DAYS_PER_YEAR * STEPS_PER_DAY
    dt = 1.0 / STEPS_PER_YEAR
    NUM_SIMULATIONS = 1000
    CONTRACT_MULTIPLIER = 3000

    STEPS_2W = int(round(2 * 5 * STEPS_PER_DAY)) 
    STEPS_3W = int(round(3 * 5 * STEPS_PER_DAY)) 

    # --- 2. Simulate 100 GBM Paths ---
    np.random.seed(67)

    Z = np.random.standard_normal((NUM_SIMULATIONS, STEPS_3W))
    drift = (- 0.5 * sigma**2) * dt
    diffusion = sigma * np.sqrt(dt) * Z
    log_returns = drift + diffusion

    prices = np.zeros((NUM_SIMULATIONS, STEPS_3W + 1))
    prices[:, 0] = S0

    for t in range(1, STEPS_3W + 1):
        prices[:, t] = prices[:, t-1] * np.exp(log_returns[:, t-1])

    prices_2w = prices[:, STEPS_2W]
    prices_3w = prices[:, STEPS_3W]

    # --- 3. Payoff Calculations ---
    payoffs = {}
    payoffs["AC"] = prices_3w  # Underlying asset
    payoffs["AC_50_P"] = np.maximum(50 - prices_3w, 0)
    payoffs["AC_50_C"] = np.maximum(prices_3w - 50, 0)
    payoffs["AC_35_P"] = np.maximum(35 - prices_3w, 0)
    payoffs["AC_40_P"] = np.maximum(40 - prices_3w, 0)
    payoffs["AC_45_P"] = np.maximum(45 - prices_3w, 0)
    payoffs["AC_60_C"] = np.maximum(prices_3w - 60, 0)
    payoffs["AC_50_P_2"] = np.maximum(50 - prices_2w, 0)
    payoffs["AC_50_C_2"] = np.maximum(prices_2w - 50, 0)

    # Chooser Option 
    is_call = prices_2w > 50
    payoffs["AC_50_CO"] = np.where(is_call, 
                            np.maximum(prices_3w - 50, 0), 
                            np.maximum(50 - prices_3w, 0))

    # Binary Put Option 
    payoffs["AC_40_BP"] = np.where(prices_3w < 40, 10, 0)

    # Knock-Out Put Option 
    min_prices_3w = np.min(prices[:, :STEPS_3W+1], axis=1)
    payoffs["AC_45_KO"] = np.where(min_prices_3w < 35, 
                                    0.0, 
                                    np.maximum(45 - prices_3w, 0))

    fairvalues = {option: np.mean(payoff) for option, payoff in payoffs.items()}

    return fairvalues

fairvalues = options_pricing()

# --- 4. Market Data Dictionary ---
market_data = {

    "AC_50_P":   {"bid_size": 50,  "bid": 12.00,  "ask": 12.05,  "ask_size": 50},
    "AC_50_C":   {"bid_size": 50,  "bid": 12.00,  "ask": 12.05,  "ask_size": 50},
    "AC_35_P":   {"bid_size": 50,  "bid": 4.33,   "ask": 4.35,   "ask_size": 50},
    "AC_40_P":   {"bid_size": 50,  "bid": 6.50,   "ask": 6.55,   "ask_size": 50},
    "AC_45_P":   {"bid_size": 50,  "bid": 9.05,   "ask": 9.10,   "ask_size": 50},
    "AC_60_C":   {"bid_size": 50,  "bid": 8.80,   "ask": 8.85,   "ask_size": 50},
    "AC_50_P_2": {"bid_size": 50,  "bid": 9.70,   "ask": 9.75,   "ask_size": 50},
    "AC_50_C_2": {"bid_size": 50,  "bid": 9.70,   "ask": 9.75,   "ask_size": 50},
    "AC_50_CO":  {"bid_size": 50,  "bid": 22.20,  "ask": 22.30,  "ask_size": 50},
    "AC_40_BP":  {"bid_size": 50,  "bid": 5.00,   "ask": 5.10,   "ask_size": 50},
    "AC_45_KO":  {"bid_size": 500, "bid": 0.15,   "ask": 0.175,  "ask_size": 500}
}

# --- 5. Expected Profit Calculator ---
print(f"{'Product':<12} | {'Fair Value':<10} | {'Action':<6} | {'Size':<6} | {'Edge/Unit':<10} | {'Expected PnL':<15}")
print("-" * 76)

total_expected_pnl = 0.0

for product, data in market_data.items():
    fv = np.mean(payoffs[product])
    bid = data["bid"]
    ask = data["ask"]
    
    action = "NONE"
    trade_size = 0
    edge = 0.0
    expected_pnl = 0.0
    
    if fv < bid:
        action = f"SELL @ {bid}"
        edge = bid - fv
        trade_size = data["bid_size"]
        expected_pnl = edge * trade_size * CONTRACT_MULTIPLIER
    elif fv > ask:
        action = f"BUY @ {ask}"
        edge = fv - ask
        trade_size = data["ask_size"]
        expected_pnl = edge * trade_size * CONTRACT_MULTIPLIER
        
    total_expected_pnl += expected_pnl
    
    print(f"{product:<12} | {fv:<10.3f} | {action:<15} | {trade_size:<6} | {edge:<10.3f} | ${expected_pnl:<14,.2f}")

print("-" * 76)
print(f"TOTAL EXPECTED PNL (Assuming 100 paths match actual): ${total_expected_pnl:,.2f}")

Product      | Fair Value | Action | Size   | Edge/Unit  | Expected PnL   
----------------------------------------------------------------------------
AC_50_P      | 12.170     | BUY @ 12.05     | 50     | 0.120      | $17,928.41     
AC_50_C      | 12.428     | BUY @ 12.05     | 50     | 0.378      | $56,679.32     
AC_35_P      | 4.475      | BUY @ 4.35      | 50     | 0.125      | $18,711.19     
AC_40_P      | 6.686      | BUY @ 6.55      | 50     | 0.136      | $20,394.14     
AC_45_P      | 9.256      | BUY @ 9.1       | 50     | 0.156      | $23,339.99     
AC_60_C      | 9.128      | BUY @ 8.85      | 50     | 0.278      | $41,729.80     
AC_50_P_2    | 10.180     | BUY @ 9.75      | 50     | 0.430      | $64,546.93     
AC_50_C_2    | 9.685      | SELL @ 9.7      | 50     | 0.015      | $2,297.26      
AC_50_CO     | 22.717     | BUY @ 22.3      | 50     | 0.417      | $62,496.90     
AC_40_BP     | 4.830      | SELL @ 5.0      | 50     | 0.170      | $25,500.00     
AC_45_KO

In [22]:
# Calculating delta for exotic options like chooser, binary, and knockout by finding the expected payoff change for a small change in the underlying price at time 0. We can use the simulated paths to estimate this by looking at how the payoff changes when we shift the initial price up and down by a small amount (e.g., 0.5 unit).
for option in ["AC_50_CO", "AC_40_BP", "AC_45_KO"]:
    original_payoff = np.mean(payoffs[option])
    
    # Shift initial price up by 0.5
    S0_up = S0 + 0.5
    # Re-run the simulation with S0_up and calculate new payoff
    # (This would involve re-simulating paths and recalculating payoffs, which is computationally intensive, so we can approximate it by looking at how the payoff changes for paths that are close to the strike price)
    
    # Shift initial price down by 0.5
    S0_down = S0 - 0.5
    # Re-run the simulation with S0_down and calculate new payoff
    
    # Approximate delta as (Payoff_up - Payoff_down) / (S0_up - S0_down)
    delta_estimate = (options_pricing(S0=S0_up)[option]- options_pricing(S0=S0_down)[option]) / (S0_up - S0_down)
    #print(delta_estimate)

    print(f"Estimated Delta for {option}: {delta_estimate:.3f}")


Estimated Delta for AC_50_CO: 0.397
Estimated Delta for AC_40_BP: -0.130
Estimated Delta for AC_45_KO: 0.009
